# Teacher CSBoost vs. cost + log-loss hybrid student (no KD)

This notebook compares **baseline teacher-augmented CSBoost** (`CSBoostClassifier` as in `churn.ipynb`) to a **hybrid student** on the **same augmented features** (teacher probability remains an **input column** only). The student is an `XGBClassifier` with a **custom objective** — **no knowledge distillation** in the loss.

**Hybrid loss.** Let \(\lambda \in [0,1]\) be `LAMBDA_COST`. With raw margin \(z\), \(p=\sigma(z)\), and hard labels \(y\):
\[
g = \lambda\, g_{\mathrm{cost}} + (1-\lambda)\, g_{\log}, \quad
h = \lambda\, h_{\mathrm{cost}} + (1-\lambda)\, h_{\log}
\]
where \((g_{\mathrm{cost}}, h_{\mathrm{cost}})\) come from the **AEC / cost-sensitive boost** objective (`empulse`), and \((g_{\log}, h_{\log})\) are the gradient/Hessian of **binary cross-entropy** to \(y\): \(g_{\log}=p-y\), \(h_{\log}=p(1-p)\).

**Student regularization:** `HYBRID_XGB_EXTRA` and `TEACHER_CSBOOST_GRID_HYBRID` (shallower `max_depth` than the baseline grid) apply only to the hybrid `XGBClassifier`.

**Datasets:** churn, gmsc, upsell (not pakdd).

**Note:** Runtime is similar to running teacher CSBoost three times (boost tuning + two student variants per fold). Reduce `N_OUTER_FOLDS` / `N_REPETITIONS` for a quick smoke test.


In [ ]:

import warnings
warnings.filterwarnings("ignore")

import time
import numpy as np
import pandas as pd
from scipy.special import expit

import sklearn
sklearn.set_config(enable_metadata_routing=True)

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import make_scorer, roc_auc_score
from sklearn.model_selection import GridSearchCV, ParameterGrid, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from xgboost import XGBClassifier

from empulse.datasets import (
    load_churn_tv_subscriptions,
    load_give_me_some_credit,
    load_upsell_bank_telemarketing,
)
from empulse.metrics import expected_cost_loss
from empulse.metrics.metric.prebuilt_metrics import make_generic_cost_metric
from empulse.metrics._loss import cy_boost_grad_hess
from empulse.models import CSBoostClassifier

RANDOM_STATE = 42
N_OUTER_FOLDS = 5
N_REPETITIONS = 2
N_INNER_FOLDS = 3
N_JOBS = 1

# λ * AEC_boost + (1-λ) * binary log loss (hard labels); no KD in the loss (see markdown)
LAMBDA_COST = 0.7

# Stronger XGB regularization for the hybrid student only (merged over grid params).
HYBRID_XGB_EXTRA = {"min_child_weight": 5, "subsample": 0.8, "colsample_bytree": 0.8}

BASELINE_BOOST_GRID = {
    "model__n_estimators": [100, 200],
    "model__learning_rate": [0.05, 0.1],
    "model__max_depth": [2, 3],
    "model__min_child_weight": [1, 3, 5],
}

TEACHER_CSBOOST_GRID = {
    "n_estimators": [100, 200],
    "learning_rate": [0.05, 0.1],
    "max_depth": [2, 3],
}

# Hybrid student: shallower trees (baseline CSBoost grid unchanged above).
TEACHER_CSBOOST_GRID_HYBRID = {
    "n_estimators": [100, 200],
    "learning_rate": [0.05, 0.1],
    "max_depth": [2],
}

DATASETS = {
    "churn": ("Churn TV", load_churn_tv_subscriptions),
    "gmsc": ("GMSC", load_give_me_some_credit),
    "upsell": ("Upsell bank", load_upsell_bank_telemarketing),
}

# Set True for a quick smoke test (1 outer fold, smaller grids).
FAST_MODE = False
if FAST_MODE:
    N_OUTER_FOLDS = 2
    N_REPETITIONS = 1
    N_INNER_FOLDS = 3
    BASELINE_BOOST_GRID = {
        "model__n_estimators": [100],
        "model__learning_rate": [0.1],
        "model__max_depth": [2],
        "model__min_child_weight": [3],
    }
    TEACHER_CSBOOST_GRID = {
        "n_estimators": [100],
        "learning_rate": [0.1],
        "max_depth": [2],
    }
    TEACHER_CSBOOST_GRID_HYBRID = dict(TEACHER_CSBOOST_GRID)
    DATASETS = {"churn": DATASETS["churn"]}



In [ ]:

def take_idx(arr, idx):
    arr = np.asarray(arr)
    if arr.ndim == 0:
        return arr.item()
    return arr[idx]


def take_rows(X, idx):
    if hasattr(X, "iloc"):
        return X.iloc[idx].copy()
    return X[idx]


def make_onehot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def build_feature_preprocessor(X_frame):
    if not isinstance(X_frame, pd.DataFrame):
        X_frame = pd.DataFrame(X_frame)
    numeric_columns = X_frame.select_dtypes(include=[np.number, "bool"]).columns.tolist()
    categorical_columns = [c for c in X_frame.columns if c not in numeric_columns]
    transformers = []
    if numeric_columns:
        numeric_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ])
        transformers.append(("num", numeric_pipe, numeric_columns))
    if categorical_columns:
        categorical_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", make_onehot_encoder()),
        ])
        transformers.append(("cat", categorical_pipe, categorical_columns))
    return ColumnTransformer(transformers=transformers, remainder="drop")


def build_stratify_key(y, fn_cost, fp_cost, n_bins=3):
    y = np.asarray(y).ravel()
    fn = np.asarray(fn_cost, dtype=float).ravel()
    fp = np.asarray(fp_cost, dtype=float).ravel()
    if fn.size == 1:
        fn = np.full(len(y), fn.item())
    if fp.size == 1:
        fp = np.full(len(y), fp.item())
    mc = np.where(y == 1, fn, fp)
    unique_vals = np.unique(mc)
    if len(unique_vals) <= n_bins:
        cost_bin = np.searchsorted(unique_vals, mc)
    else:
        boundaries = np.percentile(mc, np.linspace(0, 100, n_bins + 1)[1:-1])
        cost_bin = np.digitize(mc, boundaries)
    n_levels = int(cost_bin.max()) + 1
    return (y * n_levels + cost_bin).astype(int)


def compute_aec(y_true, y_prob, tp_cost, fp_cost, tn_cost, fn_cost):
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob, dtype=float)
    n = len(y_true)
    tp_arr = np.full(n, tp_cost) if np.isscalar(tp_cost) else np.asarray(tp_cost)
    fp_arr = np.full(n, fp_cost) if np.isscalar(fp_cost) else np.asarray(fp_cost)
    tn_arr = np.full(n, tn_cost) if np.isscalar(tn_cost) else np.asarray(tn_cost)
    fn_arr = np.full(n, fn_cost) if np.isscalar(fn_cost) else np.asarray(fn_cost)
    expected = np.where(
        y_true == 1,
        y_prob * tp_arr + (1 - y_prob) * fn_arr,
        y_prob * fp_arr + (1 - y_prob) * tn_arr,
    )
    return float(np.mean(expected))


class RoutedXGBClassifier(XGBClassifier):
    def fit(self, X, y, tp_cost=None, fp_cost=None, tn_cost=None, fn_cost=None, **fit_params):
        return super().fit(X, y, **fit_params)


def build_boost_estimator():
    model = RoutedXGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=RANDOM_STATE,
        n_jobs=1,
        verbosity=0,
        min_child_weight=3,
        subsample=0.8,
        colsample_bytree=0.8,
    )
    model = model.set_fit_request(tp_cost=True, fp_cost=True, tn_cost=True, fn_cost=True)
    return Pipeline([("preprocessor", None), ("model", model)])


def build_csboost_estimator(preprocessor):
    est = clone(preprocessor)
    model = CSBoostClassifier(
        estimator=RoutedXGBClassifier(
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=RANDOM_STATE,
            n_jobs=1,
            verbosity=0,
            min_child_weight=3,
            subsample=0.8,
            colsample_bytree=0.8,
        ),
    ).set_fit_request(tp_cost=True, fp_cost=True, tn_cost=True, fn_cost=True)
    return Pipeline([("preprocessor", est), ("model", model)])


aec_scorer = make_scorer(
    expected_cost_loss,
    greater_is_better=False,
    response_method="predict_proba",
    normalize=True,
).set_score_request(tp_cost=True, fp_cost=True, tn_cost=True, fn_cost=True)


def fit_boost_best_params(X_train, y_train, tp_train, fp_train, tn_train, fn_train, stratify_train):
    inner = list(StratifiedKFold(n_splits=N_INNER_FOLDS, shuffle=True, random_state=RANDOM_STATE).split(X_train, stratify_train))
    pipe_template = build_boost_estimator()
    pipe_template.named_steps["preprocessor"] = clone(FEATURE_PREPROCESSOR_REF)
    grid = GridSearchCV(
        pipe_template,
        BASELINE_BOOST_GRID,
        cv=inner,
        scoring=aec_scorer,
        n_jobs=N_JOBS,
        refit=True,
        error_score="raise",
    )
    grid.fit(X_train, y_train, tp_cost=tp_train, fp_cost=fp_train, tn_cost=tn_train, fn_cost=fn_train)
    best = grid.best_estimator_.named_steps["model"]
    return {
        "n_estimators": best.n_estimators,
        "max_depth": best.max_depth,
        "learning_rate": best.learning_rate,
        "min_child_weight": best.min_child_weight,
        "subsample": best.subsample,
        "colsample_bytree": best.colsample_bytree,
    }


def generate_oof_teacher_probs(X_proc, y, teacher_params, stratify):
    oof = np.full(len(y), np.nan)
    inner = StratifiedKFold(n_splits=N_INNER_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    for fi, vi in inner.split(X_proc, stratify):
        teacher = XGBClassifier(
            n_estimators=teacher_params["n_estimators"],
            max_depth=teacher_params["max_depth"],
            learning_rate=teacher_params["learning_rate"],
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=RANDOM_STATE,
            n_jobs=1,
            verbosity=0,
            min_child_weight=teacher_params.get("min_child_weight", 3),
            subsample=teacher_params.get("subsample", 0.8),
            colsample_bytree=teacher_params.get("colsample_bytree", 0.8),
        )
        teacher.fit(take_rows(X_proc, fi), y[fi])
        oof[vi] = teacher.predict_proba(take_rows(X_proc, vi))[:, 1]
    return oof


def augment_features(X, probs):
    if hasattr(X, "toarray"):
        X = X.toarray()
    return np.column_stack([X, probs])


def make_cost_logloss_hybrid_objective(y_train, tp_train, fp_train, tn_train, fn_train, lambda_cost):
    """Sklearn XGB API: objective(labels, preds). preds = raw margin z.

    g = λ g_cost + (1-λ) g_logloss,  h = λ h_cost + (1-λ) h_logloss (see markdown).
    """
    lam = float(lambda_cost)
    if not 0.0 <= lam <= 1.0:
        raise ValueError("lambda_cost must be in [0, 1].")
    metric = make_generic_cost_metric()
    gc = metric._prepare_boost_objective(
        np.asarray(y_train, dtype=float),
        tp_cost=tp_train,
        fp_cost=fp_train,
        tn_cost=tn_train,
        fn_cost=fn_train,
    ).reshape(-1)
    yi = np.asarray(y_train, dtype=np.int32)
    y_hard = yi.astype(np.float64)

    def objective(labels, preds):
        z = np.asarray(preds, dtype=float)
        g_cost, h_cost = cy_boost_grad_hess(yi, z, gc)
        p = expit(z)
        g_log = p - y_hard
        h_log = p * (1.0 - p)
        g = lam * g_cost + (1.0 - lam) * g_log
        h = lam * h_cost + (1.0 - lam) * h_log
        return g, h

    return objective


def inner_cv_teacher_csboost_aec(X_train_aug, y_train, tp_train, fp_train, tn_train, fn_train, csboost_params, stratify_train):
    aecs = []
    inner = StratifiedKFold(n_splits=N_INNER_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    for fi, vi in inner.split(X_train_aug, stratify_train):
        X_fit = take_rows(X_train_aug, fi)
        X_val = take_rows(X_train_aug, vi)
        y_fit, y_val = y_train[fi], y_train[vi]
        tp_fit = take_idx(tp_train, fi)
        fp_fit = take_idx(fp_train, fi)
        tn_fit = take_idx(tn_train, fi)
        fn_fit = take_idx(fn_train, fi)
        tp_v = take_idx(tp_train, vi)
        fp_v = take_idx(fp_train, vi)
        tn_v = take_idx(tn_train, vi)
        fn_v = take_idx(fn_train, vi)
        csb = CSBoostClassifier(
            estimator=XGBClassifier(
                **csboost_params,
                objective="binary:logistic",
                eval_metric="logloss",
                random_state=RANDOM_STATE,
                n_jobs=1,
                verbosity=0,
            ),
        ).set_fit_request(tp_cost=True, fp_cost=True, tn_cost=True, fn_cost=True)
        csb.fit(
            X_fit,
            y_fit,
            tp_cost=tp_fit,
            fp_cost=fp_fit,
            tn_cost=tn_fit,
            fn_cost=fn_fit,
        )
        y_prob = csb.predict_proba(X_val)[:, 1]
        aecs.append(compute_aec(y_val, y_prob, tp_v, fp_v, tn_v, fn_v))
    return float(np.mean(aecs))


def inner_cv_hybrid_student_aec(
    X_train_aug, y_train, tp_train, fp_train, tn_train, fn_train, xgb_params, stratify_train, lambda_cost
):
    aecs = []
    inner = StratifiedKFold(n_splits=N_INNER_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    for fi, vi in inner.split(X_train_aug, stratify_train):
        X_fit = take_rows(X_train_aug, fi)
        X_val = take_rows(X_train_aug, vi)
        y_fit, y_val = y_train[fi], y_train[vi]
        tp_fit = take_idx(tp_train, fi)
        fp_fit = take_idx(fp_train, fi)
        tn_fit = take_idx(tn_train, fi)
        fn_fit = take_idx(fn_train, fi)
        tp_v = take_idx(tp_train, vi)
        fp_v = take_idx(fp_train, vi)
        tn_v = take_idx(tn_train, vi)
        fn_v = take_idx(fn_train, vi)
        obj = make_cost_logloss_hybrid_objective(
            y_fit, tp_fit, fp_fit, tn_fit, fn_fit, lambda_cost
        )
        model = XGBClassifier(
            **{**xgb_params, **HYBRID_XGB_EXTRA},
            objective=obj,
            base_score=0.5 + 1e-2,
            random_state=RANDOM_STATE,
            n_jobs=1,
            verbosity=0,
        )
        model.fit(X_fit, y_fit)
        y_prob = model.predict_proba(X_val)[:, 1]
        aecs.append(compute_aec(y_val, y_prob, tp_v, fp_v, tn_v, fn_v))
    return float(np.mean(aecs))


FEATURE_PREPROCESSOR_REF = None



In [ ]:

def run_comparison_for_dataset(name, load_fn):
    global FEATURE_PREPROCESSOR_REF
    print("=" * 72)
    print(f"Dataset: {name}")
    print("=" * 72)
    dataset = load_fn(as_frame=True)
    X = dataset.data.copy()
    y = dataset.target.copy()
    tp_cost = dataset.tp_cost
    fp_cost = dataset.fp_cost
    tn_cost = dataset.tn_cost
    fn_cost = dataset.fn_cost

    X_data = X.copy() if hasattr(X, "copy") else pd.DataFrame(X)
    FEATURE_PREPROCESSOR_REF = build_feature_preprocessor(X_data)
    y_arr = y.values if hasattr(y, "values") else np.asarray(y)
    tp_arr = np.asarray(tp_cost)
    fp_arr = np.asarray(fp_cost)
    tn_arr = np.asarray(tn_cost)
    fn_arr = np.asarray(fn_cost)
    stratify_arr = build_stratify_key(y_arr, fn_arr, fp_arr)

    INNER_CV = StratifiedKFold(n_splits=N_INNER_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    cv_splits = []
    for rep in range(N_REPETITIONS):
        outer_cv = StratifiedKFold(n_splits=N_OUTER_FOLDS, shuffle=True, random_state=RANDOM_STATE + rep)
        for fold, (train_idx, test_idx) in enumerate(outer_cv.split(X_data, stratify_arr), start=1):
            cv_splits.append({"rep": rep + 1, "fold": fold, "train_idx": train_idx, "test_idx": test_idx})

    if FAST_MODE:
        cv_splits = cv_splits[:1]

    rows_base = []
    rows_dis = []

    for split in cv_splits:
        train_idx, test_idx = split["train_idx"], split["test_idx"]
        X_train = take_rows(X_data, train_idx)
        X_test = take_rows(X_data, test_idx)
        y_train, y_test = y_arr[train_idx], y_arr[test_idx]
        tp_train = take_idx(tp_arr, train_idx)
        fp_train = take_idx(fp_arr, train_idx)
        tn_train = take_idx(tn_arr, train_idx)
        fn_train = take_idx(fn_arr, train_idx)
        tp_test = take_idx(tp_arr, test_idx)
        fp_test = take_idx(fp_arr, test_idx)
        tn_test = take_idx(tn_arr, test_idx)
        fn_test = take_idx(fn_arr, test_idx)
        stratify_train = take_idx(stratify_arr, train_idx)

        preprocessor = clone(FEATURE_PREPROCESSOR_REF)
        X_train_proc = preprocessor.fit_transform(X_train)
        X_test_proc = preprocessor.transform(X_test)

        teacher_params = fit_boost_best_params(
            X_train, y_train, tp_train, fp_train, tn_train, fn_train, stratify_train
        )
        oof_teacher = generate_oof_teacher_probs(X_train_proc, y_train, teacher_params, stratify_train)
        full_teacher = XGBClassifier(
            n_estimators=teacher_params["n_estimators"],
            max_depth=teacher_params["max_depth"],
            learning_rate=teacher_params["learning_rate"],
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=RANDOM_STATE,
            n_jobs=1,
            verbosity=0,
            min_child_weight=teacher_params.get("min_child_weight", 3),
            subsample=teacher_params.get("subsample", 0.8),
            colsample_bytree=teacher_params.get("colsample_bytree", 0.8),
        )
        full_teacher.fit(X_train_proc, y_train)
        test_teacher = full_teacher.predict_proba(X_test_proc)[:, 1]
        X_train_aug = augment_features(X_train_proc, oof_teacher)
        X_test_aug = augment_features(X_test_proc, test_teacher)

        best_params, best_aec_base = None, float("inf")
        for params in ParameterGrid(TEACHER_CSBOOST_GRID):
            aec = inner_cv_teacher_csboost_aec(
                X_train_aug, y_train, tp_train, fp_train, tn_train, fn_train, params, stratify_train
            )
            if aec < best_aec_base:
                best_params, best_aec_base = dict(params), aec

        best_hybrid_params, best_aec_hybrid = None, float("inf")
        for params in ParameterGrid(TEACHER_CSBOOST_GRID_HYBRID):
            aec = inner_cv_hybrid_student_aec(
                X_train_aug,
                y_train,
                tp_train,
                fp_train,
                tn_train,
                fn_train,
                params,
                stratify_train,
                LAMBDA_COST,
            )
            if aec < best_aec_hybrid:
                best_hybrid_params, best_aec_hybrid = dict(params), aec

        final_base = CSBoostClassifier(
            estimator=XGBClassifier(
                **best_params,
                objective="binary:logistic",
                eval_metric="logloss",
                random_state=RANDOM_STATE,
                n_jobs=1,
                verbosity=0,
            ),
        ).set_fit_request(tp_cost=True, fp_cost=True, tn_cost=True, fn_cost=True)
        final_base.fit(
            X_train_aug,
            y_train,
            tp_cost=tp_train,
            fp_cost=fp_train,
            tn_cost=tn_train,
            fn_cost=fn_train,
        )
        prob_base = final_base.predict_proba(X_test_aug)[:, 1]

        obj_final = make_cost_logloss_hybrid_objective(
            y_train,
            tp_train,
            fp_train,
            tn_train,
            fn_train,
            LAMBDA_COST,
        )
        final_hybrid = XGBClassifier(
            **{**best_hybrid_params, **HYBRID_XGB_EXTRA},
            objective=obj_final,
            base_score=0.5 + 1e-2,
            random_state=RANDOM_STATE,
            n_jobs=1,
            verbosity=0,
        )
        final_hybrid.fit(X_train_aug, y_train)
        prob_hybrid = final_hybrid.predict_proba(X_test_aug)[:, 1]

        rows_base.append({
            "rep": split["rep"],
            "fold": split["fold"],
            "AEC": compute_aec(y_test, prob_base, tp_test, fp_test, tn_test, fn_test),
            "AUROC": roc_auc_score(y_test, prob_base),
        })
        rows_dis.append({
            "rep": split["rep"],
            "fold": split["fold"],
            "AEC": compute_aec(y_test, prob_hybrid, tp_test, fp_test, tn_test, fn_test),
            "AUROC": roc_auc_score(y_test, prob_hybrid),
        })

    df_b = pd.DataFrame(rows_base)
    df_d = pd.DataFrame(rows_dis)
    summ = pd.DataFrame({
        "teacher_csboost (baseline)": [df_b["AEC"].mean(), df_b["AEC"].std(), df_b["AUROC"].mean(), df_b["AUROC"].std()],
        f"teacher_csboost + λ·cost+(1-λ)·logloss (λ={LAMBDA_COST})": [
            df_d["AEC"].mean(), df_d["AEC"].std(), df_d["AUROC"].mean(), df_d["AUROC"].std()],
    }, index=["AEC_mean", "AEC_std", "AUROC_mean", "AUROC_std"])
    print(summ.round(6))
    print(f"\nMean AEC delta (hybrid - baseline): {df_d['AEC'].mean() - df_b['AEC'].mean():+.6f}")
    print(f"Mean AUROC delta (hybrid - baseline): {df_d['AUROC'].mean() - df_b['AUROC'].mean():+.6f}")
    return summ, df_b, df_d



In [ ]:

all_summaries = {}
for key, (title, loader) in DATASETS.items():
    summ, _, _ = run_comparison_for_dataset(title, loader)
    all_summaries[key] = summ

summary_table = pd.concat(all_summaries, axis=1)
print("\n" + "#" * 72)
print("Combined summary (mean/std over outer folds)")
print("#" * 72)
print(summary_table.round(6))

